# Strands Multi-Agent System with AgentCore Memory (Short Term Memory) - Latest SDK

**Updated for Latest AgentCore SDK** — This notebook keeps the custom hooks pattern but migrates from the legacy `MemoryClient` API to the recommended `MemorySessionManager` / `MemorySession` / `ConversationalMessage` APIs.

## Introduction

This notebook demonstrates how to implement a **multi-agent system with shared memory** using AWS AgentCore Memory and the Strands framework with **custom hooks**.

The original notebook (`travel-planning-agent.ipynb`) used the legacy `MemoryClient` with tuple-based messages and `create_event()`. This updated version migrates the hook implementation to use:
- `MemorySessionManager` → recommended primary interface for managing sessions
- `MemorySession` → session-scoped operations (no need to pass memory_id/actor_id/session_id repeatedly)
- `ConversationalMessage` / `MessageRole` → type-safe message objects instead of tuples
- `add_turns()` → replaces `create_event()`

This approach is useful when you need **custom control** over how memory is loaded and saved (e.g., custom formatting, filtering, or conditional logic).

For a simpler approach that eliminates hooks entirely, see `travel-planning-agent-memory-manager_latest_sdk.ipynb` which uses `AgentCoreMemorySessionManager`.

## Tutorial Details

| Information         | Details                                                                          |
|:--------------------|:---------------------------------------------------------------------------------|
| Tutorial type       | Short Term Conversational                                                        |
| Agent usecase       | Travel Planning Assistant                                                        |
| Agentic Framework   | Strands Agents                                                                   |
| LLM model           | Anthropic Claude Haiku 4.5                                                       |
| Tutorial components | AgentCore Short-term Memory, Strands Agents, Custom HookProvider, MemorySessionManager |
| Example complexity  | Intermediate                                                                     |

What you will learn:

- How to use `MemorySessionManager` and `MemorySession` for session-scoped memory operations
- Implementing custom hooks with `ConversationalMessage` and `MessageRole`
- Creating specialized agents with shared memory
- Implementing a coordinator agent that delegates to specialized agents

### Scenario context

In this example, we'll create a **Travel Planning System** with:
1. A Flight Booking Assistant specialized in air travel
2. A Hotel Booking Assistant focused on accommodations
3. A Travel Coordinator that delegates to these specialized agents

## Architecture
<div style="text-align:left">
    <img src="architecture.png" width="65%" />
</div>

## Prerequisites
- Python 3.10+
- AWS account with appropriate permissions
- AWS IAM role with appropriate permissions for AgentCore Memory
- Access to Amazon Bedrock models

## Step 1: Environment Setup
Install dependencies and import the necessary libraries.

In [ ]:
!pip install -qr requirements.txt

In [ ]:
import logging
import os
from datetime import datetime

from bedrock_agentcore.memory import MemoryClient, MemorySessionManager
from bedrock_agentcore.memory.constants import ConversationalMessage, MessageRole
from strands import Agent, tool
from strands.hooks import AgentInitializedEvent, HookProvider, HookRegistry, MessageAddedEvent

In [ ]:
region = os.getenv('AWS_REGION', 'us-west-2')
MODEL_ID = "global.anthropic.claude-haiku-4-5-20251001-v1:0"

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s", datefmt="%Y-%m-%d %H:%M:%S")
logger = logging.getLogger("agentcore-memory")

## Step 2: Creating Shared Memory
Create a memory resource that will be shared among our specialized agents.

In [ ]:
client = MemoryClient(region_name=region)

print("Creating Memory...")
memory_name = f"TravelAgent_STM_{datetime.now().strftime('%Y%m%d%H%M%S')}"

memory = client.create_memory(
    name=memory_name,
    description="Short-term memory for travel agent"
)

memory_id = memory['id']
logger.info(f"Memory ready: {memory_id}")

## Step 3: Initialize MemorySessionManager

`MemorySessionManager` is the recommended primary interface for managing sessions. It replaces direct `MemoryClient` calls for session-scoped operations.

In [ ]:
session_manager = MemorySessionManager(memory_id=memory_id, region_name=region)
logger.info(f"MemorySessionManager initialized for memory: {memory_id}")

## Step 4: Configure Actor and Session IDs

The `actorId` represents the **user** identity, not the agent. Using a consistent `actorId` ensures memory persists across sessions.

In [ ]:
# actorId represents the USER identity - consistent across sessions for memory persistence
user_actor_id = "user-001"
session_id = f"travel-session-{datetime.now().strftime('%Y%m%d%H%M%S')}"

# Both agents use the same user actorId
flight_actor_id = user_actor_id
hotel_actor_id = user_actor_id

logger.info(f"User Actor ID: {user_actor_id}")
logger.info(f"Shared Session ID: {session_id}")

## Step 5: Create Custom Memory Hook Provider

This hook demonstrates **custom control** over memory operations using the latest SDK APIs.

**Key changes from the original:**
- Uses `MemorySession` instead of `MemoryClient` (no need to pass memory_id/actor_id/session_id to every call)
- Uses `ConversationalMessage` + `MessageRole` instead of raw tuples
- Uses `add_turns()` instead of `create_event()`
- Uses `get_last_k_turns()` on the session object directly

In [ ]:
class ShortTermMemoryHook(HookProvider):
    """Custom hook that uses MemorySession for automatic memory load/save."""

    def __init__(self, memory_session):
        self.memory_session = memory_session

    def on_agent_initialized(self, event: AgentInitializedEvent):
        """Load recent conversation history when agent starts."""
        try:
            recent_turns = self.memory_session.get_last_k_turns(k=5)

            if recent_turns:
                context_messages = []
                for turn in recent_turns:
                    for message in turn:
                        role = message.get('role', 'unknown')
                        content = message.get('content', {}).get('text', '')
                        context_messages.append(f"{role}: {content}")

                context = "\n".join(context_messages)
                event.agent.system_prompt += (
                    f"\n\nRecent conversation history:\n{context}"
                    f"\n\nContinue the conversation naturally based on this context."
                )
                logger.info(f"Loaded {len(recent_turns)} recent conversation turns")
            else:
                logger.info("No previous conversation history found")

        except Exception as e:
            logger.error(f"Failed to load conversation history: {e}")

    def on_message_added(self, event: MessageAddedEvent):
        """Store messages in memory using ConversationalMessage."""
        messages = event.agent.messages
        try:
            if messages and messages[-1]["content"][0].get("text"):
                message_text = messages[-1]["content"][0]["text"]
                message_role = (
                    MessageRole.USER if messages[-1]["role"] == "user"
                    else MessageRole.ASSISTANT
                )

                self.memory_session.add_turns(
                    messages=[ConversationalMessage(message_text, message_role)]
                )
                logger.info(f"Stored message with role: {message_role.value}")

        except Exception as e:
            logger.error(f"Memory save error: {e}")

    def register_hooks(self, registry: HookRegistry) -> None:
        registry.add_callback(AgentInitializedEvent, self.on_agent_initialized)
        registry.add_callback(MessageAddedEvent, self.on_message_added)

## Step 6: Define System Prompts

In [ ]:
HOTEL_BOOKING_PROMPT = """You are a hotel booking assistant. Help customers find hotels, make reservations, and answer questions about accommodations and amenities.
Provide clear information about availability, pricing, and booking procedures in a friendly, helpful manner.
Keep the messages short, don't overwhelm the customer."""

FLIGHT_BOOKING_PROMPT = """You are a flight booking assistant. Help customers find flights, make reservations, and answer questions about airlines, routes, and travel policies.
Provide clear information about flight availability, pricing, schedules, and booking procedures in a friendly, helpful manner.
Keep the messages short, don't overwhelm the customer."""

## Step 7: Implementing Agent Tools with Hooks

Each tool creates a `MemorySession` via `session_manager.create_memory_session()` and passes it to the hook. The session object encapsulates memory_id, actor_id, and session_id so the hook doesn't need to manage them.

In [ ]:
@tool
def flight_booking_assistant(query: str) -> str:
    """
    Process and respond to flight booking queries.

    Args:
        query: A flight-related question about bookings, schedules, airlines, or travel policies

    Returns:
        Detailed flight information, booking options, or travel advice
    """
    try:
        memory_session = session_manager.create_memory_session(
            actor_id=flight_actor_id,
            session_id=session_id
        )
        flight_hook = ShortTermMemoryHook(memory_session)

        flight_agent = Agent(
            hooks=[flight_hook],
            model=MODEL_ID,
            system_prompt=FLIGHT_BOOKING_PROMPT
        )

        response = flight_agent(query)
        return str(response)

    except Exception as e:
        logger.error(f"Error in flight booking assistant: {e}")
        return f"Error in flight booking assistant: {str(e)}"


@tool
def hotel_booking_assistant(query: str) -> str:
    """
    Process and respond to hotel booking queries.

    Args:
        query: A hotel-related question about accommodations, amenities, or reservations

    Returns:
        Detailed hotel information, booking options, or accommodation advice
    """
    try:
        memory_session = session_manager.create_memory_session(
            actor_id=hotel_actor_id,
            session_id=session_id
        )
        hotel_hook = ShortTermMemoryHook(memory_session)

        hotel_agent = Agent(
            hooks=[hotel_hook],
            model=MODEL_ID,
            system_prompt=HOTEL_BOOKING_PROMPT
        )

        response = hotel_agent(query)
        return str(response)

    except Exception as e:
        logger.error(f"Error in hotel booking assistant: {e}")
        return f"Error in hotel booking assistant: {str(e)}"

## Step 8: Creating the Coordinator Agent

In [ ]:
TRAVEL_AGENT_SYSTEM_PROMPT = """
You are a comprehensive travel planning assistant that coordinates between specialized tools:
- For flight-related queries (bookings, schedules, airlines, routes) → Use the flight_booking_assistant tool
- For hotel-related queries (accommodations, amenities, reservations) → Use the hotel_booking_assistant tool
- For complete travel packages → Use both tools as needed to provide comprehensive information
- For general travel advice or simple travel questions → Answer directly

Each agent will have its own memory in case the user asks about historic data.
When handling complex travel requests, coordinate information from both tools to create a cohesive travel plan.
Provide clear organization when presenting information from multiple sources.
Keep the messages short, don't overwhelm the customer.
"""

In [ ]:
travel_agent = Agent(
    system_prompt=TRAVEL_AGENT_SYSTEM_PROMPT,
    model=MODEL_ID,
    tools=[flight_booking_assistant, hotel_booking_assistant]
)

logger.info("Travel coordinator agent created")

## Step 9: Test the Multi-Agent System

In [ ]:
response = travel_agent("Hello, I would like to book a trip from LA to Madrid. From July 1 to August 2.")
print(response)

In [ ]:
response = travel_agent("I would only like to focus on the flight at the moment. Direct flights preferred, economy class.")
print(response)

In [ ]:
response = travel_agent("Now let's look at hotels. I prefer mid-range, city center, with a pool.")
print(response)

## Step 10: Testing Memory Persistence

Create a new coordinator instance to verify the specialized agents remember previous conversations:

In [ ]:
new_travel_agent = Agent(
    system_prompt=TRAVEL_AGENT_SYSTEM_PROMPT,
    model=MODEL_ID,
    tools=[flight_booking_assistant, hotel_booking_assistant]
)

response = new_travel_agent("Can you remind me about the flights we discussed?")
print(response)

## Summary

In this notebook, we've demonstrated:

1. How to create a shared memory resource for multiple agents
2. How to implement custom hooks using `MemorySession`, `ConversationalMessage`, and `add_turns()`
3. How to coordinate between multiple agents while maintaining conversation context
4. How memory persists across different agent instances

This approach keeps the hooks pattern for custom control while migrating from the legacy `MemoryClient` API to the recommended `MemorySessionManager` / `MemorySession` pattern.

## Clean up
Delete the memory to clean up resources:

In [ ]:
# Uncomment to delete memory resource
# try:
#     client.delete_memory(memory_id=memory_id)
#     logger.info(f"Deleted memory: {memory_id}")
# except Exception as e:
#     logger.error(f"Failed to delete memory: {e}")